# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Freestyle submission, V2 edition.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* The pipeline was rebuilt as **V2**: the whole process was written down and hashed in `work/V2_SPEC.md` before any V2 code ran. V1 stays in the repository as the first process, and every V1 number shown here is labelled V1. My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** ranked actions with reason codes, an archetype-to-action map, the key decay insight, intended
use, limits, human review, cost and value, monitoring and retrain triggers, and what must not be automated. **The
principle it serves:** rules give the reason code, models give the order, and a person decides.

In [1]:
# Setup: works locally and in Colab. Reads committed receipts; the warehouse is touched only by the opt-in rerun.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
RERUN_PIPELINE = False   # opt-in: True recomputes from the pinned snapshot (needs HF_TOKEN). V2 steps then run in a scratch copy, never in the repo.
DATA = wh.data_available(try_download=RERUN_PIPELINE)
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8], "| RERUN_PIPELINE =", RERUN_PIPELINE)
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

# V2 receipts (aggregates only) and the helpers every notebook shares.
v2 = lambda n: receipt(f"v2_{n}.json")
BASE = ["rule_position_slipping", "mean_reversion_ctr_excess", "client_shock", "persistence_prev_ratio"]
BASE_LABEL = {"rule_position_slipping": "editor's rule", "mean_reversion_ctr_excess": "CTR above peers",
              "client_shock": "client shock", "persistence_prev_ratio": "persistence"}
def v2_months():
    """(month, role, evaluation block) for the four evaluated months. The logistic champion is in every block."""
    vf, cf, ts = v2("validation_folds"), v2("confirmation_apr"), v2("test")
    return [("2026-02", "validation", vf["folds"]["validation_1"]), ("2026-03", "validation", vf["folds"]["validation_2"]),
            ("2026-04", "confirmation", cf["evaluation"]), ("2026-05", "test", ts["evaluation"])]
def champ(ev):
    return ev["models"].get("LOGISTIC") or ev["models"]["SELECTED:LOGISTIC"]
def fold_table():
    """One row per evaluated month: the champion against the four frozen baselines, same rows, same metric."""
    rows = []
    for m, role, ev in v2_months():
        lg, best = champ(ev), ev["best_baseline_by_ap"]
        rows.append({"month": m, "role": role, "base rate": ev["base_rate"], "rule P@50": ev["models"][BASE[0]]["p@50"],
                     "best baseline (by AP)": BASE_LABEL[best], "best baseline AP": ev["models"][best]["ap"],
                     "V2 AP": lg["ap"], "V2 AP 95% CI": lg["ap_ci"], "V2 P@50": lg["p@50"], "V2 P@50 95% CI": lg["p@50_ci"],
                     "P@50, unmeasurable = miss": lg["p@50_unmeasurable_as_miss"],
                     "rule P@50, unmeasurable = miss": ev["models"][BASE[0]]["p@50_unmeasurable_as_miss"],
                     "rule share of click shortfall (top 50)": ev["models"][BASE[0]]["clicks_lost_captured"],
                     "V2 share of click shortfall (top 50)": lg["clicks_lost_captured"]})
    return pd.DataFrame(rows)

pinned warehouse snapshot available: True | revision 50cbf7c3 | RERUN_PIPELINE = False


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Question:** what should an editor do first, and why? **Why it matters (FlyRank principle: rules give the reason code,
models give the order):** the queue has to be explainable in words a human trusts.

**The ranked queue for the July 2026 decision.** The V2 selected model (the logistic champion, because no challenger passed the
rule), refit on every month whose label had matured (December to May, 49,022 rows, 36 features), scores 14,100 eligible pages.
Pages younger than 90 days are excluded (3,243) and no client gets more than 10 slots. The reason code is the largest positive standardised
contribution of the champion logistic fitted on the same rows, mapped to a plain-language family. Rows are shown by rank and an anonymous
client index. `rank` is the model's rank before the guardrails, so gaps are pages removed for being younger than 90 days or for exceeding a
client's cap. The full queue with IDs for the editor is written to a gitignored CSV; its SHA-256 is recorded in `v2_playbook_summary.json`, which
makes the queue a frozen artefact: any later change to the file changes the hash.

In [2]:
pb = v2("playbook_summary")
print({k: pb[k] for k in ["decision_origin", "feature_window", "trained_on_origins", "train_rows", "eligible_pages", "excluded_younger_than_90_days",
                          "queue_size", "queue_clients", "max_pages_one_client", "reason_codes", "clicks_at_risk_2_months_if_25pct_lost"]})
print("reason codes over all eligible pages:", pb["reason_codes_all_eligible"])
print("queue pages also in the V1 queue:", pb["overlap_with_v1_queue_pages"], "of", pb["queue_size"])

{'decision_origin': '2026-07', 'feature_window': ['2026-05', '2026-06'], 'trained_on_origins': ['2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05'], 'train_rows': 49022, 'eligible_pages': 14100, 'excluded_younger_than_90_days': 3243, 'queue_size': 50, 'queue_clients': 7, 'max_pages_one_client': 10, 'reason_codes': {'CTR_ABOVE_PEERS': 49, 'CTR_ALREADY_SLIPPING': 1}, 'clicks_at_risk_2_months_if_25pct_lost': 25242}
reason codes over all eligible pages: {'PAGE_PROFILE': 8235, 'CTR_ABOVE_PEERS': 4614, 'CTR_ALREADY_SLIPPING': 947, 'DEMAND_SHIFT': 258, 'CLIENT_WIDE_DECLINE': 37, 'STALE_ACTIVITY': 9}
queue pages also in the V1 queue: 47 of 50


In [3]:
import hashlib
path = os.path.join(OUT, pb["queue_file"]["name"])
if os.path.exists(path):
    q = pd.read_csv(path)
    print("frozen queue file matches the recorded SHA-256:", hashlib.sha256(open(path, "rb").read()).hexdigest() == pb["queue_file"]["sha256"])
    view = q[q.in_queue].copy(); view["client"] = pd.factorize(view.client_hash_id)[0]
    display(view[["rank", "client", "position", "score", "top_driver", "reason_code"]].head(15).reset_index(drop=True))
else:
    print("the queue CSV is gitignored and not on this machine; recorded SHA-256:", pb["queue_file"]["sha256"][:16], "...")

frozen queue file matches the recorded SHA-256: True


,rank,client,position,score,top_driver,reason_code
0,7,0,2.8,0.989678,ctr_vs_ref,CTR_ABOVE_PEERS
1,8,0,2.8,0.989646,ctr_vs_ref,CTR_ABOVE_PEERS
2,9,0,2.8,0.988013,ctr_vs_ref,CTR_ABOVE_PEERS
3,15,1,4.1,0.967248,ctr_vs_ref,CTR_ABOVE_PEERS
4,16,1,4.1,0.967108,ctr_vs_ref,CTR_ABOVE_PEERS
5,17,1,4.1,0.965447,ctr_vs_ref,CTR_ABOVE_PEERS
6,18,1,4.1,0.964103,ctr_vs_ref,CTR_ABOVE_PEERS
7,19,1,4.1,0.964082,ctr_vs_ref,CTR_ABOVE_PEERS
8,20,1,4.1,0.963511,ctr_vs_ref,CTR_ABOVE_PEERS
9,21,1,4.1,0.963252,ctr_vs_ref,CTR_ABOVE_PEERS


**Reason code to action.**

| Reason code (largest model contribution) | What it means | Action |
|---|---|---|
| `CTR_ABOVE_PEERS` | the page earns more clicks than its position normally does | **Protect:** check the live results page for an AI answer; make the page the citable source (answer-first summary, FAQ, structured data) before editing anything else |
| `DEMAND_SHIFT` | impressions swung sharply, or the peak was very recent (`days_since_peak`, new in V2) | **Check demand first:** seasonality or tracking may explain the change; no rewrite until confirmed |
| `CTR_ALREADY_SLIPPING` | a CTR fall has started | rewrite title and description toward the click an AI answer cannot satisfy (comparison, pricing, tool, example) |
| `STALE_ACTIVITY` (new in V2) | a long quiet spell (`days_since_last_click`, `longest_zero_click_run`) | check whether the page still earns clicks for its query; re-read the live results page and refresh the content |
| `WEEKDAY_PROFILE` (new in V2) | traffic is concentrated on working days (`weekday_share`) | check whether the query is a work-task question an AI answer can resolve; test an answer-first rewrite |
| `TOP_POSITION_EXPOSED` | rank alone drives the score | make the answer citable; monitor |
| `RANK_MOVING` | position is unstable | **hold:** re-score next month |
| `CLIENT_WIDE_DECLINE` | the whole site is losing CTR at a stable rank | escalate to the account lead; page edits alone will not fix it |
| `PAGE_PROFILE` | no single driver | editor judgement: check intent match and freshness |

**What the July queue looks like.** 49 of the 50 queued pages are `CTR_ABOVE_PEERS` and 1 is `CTR_ALREADY_SLIPPING`; across all 14,100 eligible
pages the codes are spread (8,235 `PAGE_PROFILE`, 4,614 `CTR_ABOVE_PEERS`, 947 `CTR_ALREADY_SLIPPING`, 258 `DEMAND_SHIFT`,
37 `CLIENT_WIDE_DECLINE`, 9 `STALE_ACTIVITY`). So inside the queue the order comes from the model, and the code mainly tells the editor where to look first.
47 of the 50 queued pages are also in V1's July queue.

**The insight behind the playbook.** The pages that lose clicks next are not the ones already slipping. They are the
ones **currently earning more than their rank normally earns**. That excess click share is what an AI answer can take.
So the first action is protective: make the page the source an AI answer cites. It is not a rewrite after the loss.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Question:** who uses this, for what, and where does it stop being valid? **Why it matters (FlyRank principle: a person
decides):** a tool used outside its intended use can mislead.

- **Who:** a FlyRank content editor, once a month, with a budget of 50 reviews.
- **When:** day 1 of the month, using the two closed months before it.
- **What it is:** a decision-support queue of top-10 pages most likely to lose at least 25% of their expected clicks at
  their current rank in the next two months, each with one reason code and one suggested action.
- **What it is not:** an automatic rewrite, a forecast of Google, or a causal claim that the action restores clicks.
- **Where it stops being valid:** pages outside the top 10, pages younger than 90 days, pages whose visibility has collapsed (the
  label cannot be measured for them), and any month whose base rate leaves the range seen so far (see the monitoring triggers).

**Cost and value.**

In [4]:
pb = v2("playbook_summary"); ts = v2("test")["evaluation"]; g = ts["models"]["SELECTED:LOGISTIC"]
print(f"Test month: {g['p@50'] * 50:.0f} of 50 queued pages eroded with the model, {g['p@50_unmeasurable_as_miss'] * 50:.0f} if unmeasurable pages count as misses, against about {ts['base_rate'] * 50:.0f} for a random 50.")
print(f"Editor time: 50 reviews x about 20 minutes = about 17 hours per month.")
print(f"Clicks at stake in the July queue if each page lost 25% over two months: about {round(pb['clicks_at_risk_2_months_if_25pct_lost'], -3):,.0f}.")
display(pd.DataFrame(pb["cap_cost_descriptive"]).T)
print(pb["guardrails"])

Test month: 26 of 50 queued pages eroded with the model, 17 if unmeasurable pages count as misses, against about 12 for a random 50.
Editor time: 50 reviews x about 20 minutes = about 17 hours per month.
Clicks at stake in the July queue if each page lost 25% over two months: about 25,000.


,uncapped_p@50,capped_p@50
validation_1,0.56,0.56
validation_2,0.70,0.70
confirmation,0.60,0.58
test,0.52,0.54


{'min_age_days': 90, 'client_cap': 10, 'queue_size': 50, 'cap_in_selection_or_headline': False, 'note': 'the cap was introduced in V1 after the sealed month was opened; descriptive only here'}


**Reading it.**
- **Value of the model:** on the test month, the queue holds about 14 more eroding pages per 50 reviews than random selection (26 against about 12). Counting the pages
  that later lost visibility as misses, the gain shrinks to about 5. That is the time saved, with a wide interval (P@50 95% CI 0.32 to 0.72).
- **The 10-pages-per-client cap is a guardrail, not a result.** It was introduced in V1 after the sealed month had been opened, for fairness of the review budget, and it plays no part in any V2 selection or headline metric. Descriptively, it
  changes the champion's P@50 by -0.02 to +0.02 across the four evaluated months.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Question:** what must a person check before acting, and what should never be automated? **Why it matters (FlyRank
principle: a person decides):** every false alarm costs an edit that can put a page's rank at risk.

**Human review (required before any action).**
1. Open the live results page for the page's main query and record whether an AI answer appears and whether the page is
   cited.
2. **Verify visibility.** Confirm the page still ranks and still earns impressions. A page whose visibility has collapsed cannot be scored by the label:
   4.13% to 16.09% of eligible pages per month are unmeasurable, and counting such pages as misses lowers the test month's P@50 from 0.52 to 0.34. This is the main guardrail against acting on a
   page the model ranked for the wrong reason.
3. If impressions jumped threefold or more, or the page had data in only one month, confirm the demand is real before
   acting.
4. Respect a 30-day cooldown after any edit, and skip pages younger than 90 days.
5. The editor can override any row. Overrides are logged with a reason, and they become training signal.

**The no-go list (what must not be automated).**
- **Rewriting or publishing content from the queue.** A person reviews the live results page first.
- **Treating a false alarm as harmless.** Every edit risks the rank the page holds.
- **Claiming that an action restored clicks.** That needs a hold-out (for example, leave 10 of the 50 queued pages
  untouched each month and compare), and this study does not have one yet.
- **Client-level conclusions from page queues.** Client-wide declines go to the account lead.

In [5]:
pb = v2("playbook_summary")
print(f"guardrails in the July queue: client cap = {pb['guardrails']['client_cap']} slots, pages younger than 90 days excluded = {pb['excluded_younger_than_90_days']:,}")
print("unmeasurable share of eligible pages by evaluated month:", {m: ev["lost_visibility_share"] for m, _, ev in v2_months()})

guardrails in the July queue: client cap = 10 slots, pages younger than 90 days excluded = 3,243
unmeasurable share of eligible pages by evaluated month: {'2026-02': 0.0717, '2026-03': 0.0413, '2026-04': 0.092, '2026-05': 0.1609}


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Question:** what would tell me the recommendations went stale? **Why it matters (FlyRank principle: a model is a
process, not a file):** a ranking trained on past months drifts as Google and the panel change.

| Trigger | Threshold | Response |
|---|---|---|
| A new month's label matures | monthly | refit the frozen configuration on all matured months and re-run the forward check; a different model needs a new pre-registered spec |
| Precision of the last matured queue | below 1.3x that month's base rate (the four evaluated months run 1.95x to 2.46x) | pause the queue and investigate drift |
| Base rate moves | outside 0.15 to 0.45 (seen so far: 0.23 to 0.34) | check for a Google change or a tracking break before trusting scores |
| Share of unmeasurable pages | above the highest seen so far (16.09%) | check for a tracking break; re-read the headline with unmeasurable pages counted as misses |
| Client concentration | one client above 10 slots before the cap, or effective clients below the 4.42 seen so far | review that client's data feed; widen the intervals before reading anything |
| AI-answer presence | editors record AI answers on reviewed pages | after 3 months, test whether recorded presence adds signal |

**Next iteration (to be pre-registered before it runs).** The data ends on 2026-06-30, so the July queue cannot be scored yet. It is frozen by its SHA-256,
and the one untouched test left is its own outcome (July and August 2026), once a later data release exists.
1. **Score the frozen July queue** against its matured outcome, once, with the same metrics.
2. **Log editors' AI-answer observations as a feature.**
3. **Run the 10-page hold-out to measure the effect of the actions.**

In [6]:
print("precision-to-base-rate ratio (lift) of the champion in each evaluated month (the trigger fires below 1.3):")
display(pd.Series({m: round(champ(ev)["lift"], 2) for m, _, ev in v2_months()}).to_frame("P@50 / base rate"))

precision-to-base-rate ratio (lift) of the champion in each evaluated month (the trigger fires below 1.3):


,P@50 / base rate
2026-02,2.46
2026-03,2.07
2026-04,1.95
2026-05,2.17


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**Question:** which files does the paper build on? **Why it matters (FlyRank principle: every number traces to a
receipt):** the paper and the capstone notebook read these files, so they must exist and match.

`work/outputs/v2_queue_2026-07.csv` (gitignored, row level, frozen by its SHA-256), `v2_playbook_summary.json`, `v2_selection.json`,
`v2_validation_folds.json`, `v2_confirmation_apr.json`, `v2_test.json`, `v2_freeze.json`, `v2_leakage_checks.json`, `v2_secondary_label.json`,
`v2_features.json`, `v2_summary.json`, the V1 receipts it compares with, and the figures in `work/figures/` feed the paper.

In [7]:
for f in ["v2_spec.json", "v2_spec_hash.json", "v2_features.json", "v2_selection.json", "v2_validation_folds.json", "v2_confirmation_apr.json", "v2_secondary_label.json",
          "v2_leakage_checks.json", "v2_freeze.json", "v2_test.json", "v2_playbook_summary.json", "v2_summary.json", "v2_queue_2026-07.csv"]:
    print(f"{f:34s} exists: {os.path.exists(os.path.join(OUT, f))}")
print("figures:", sorted(f for f in os.listdir(FIG) if f.endswith(".png")))

v2_spec.json                       exists: True
v2_spec_hash.json                  exists: True
v2_features.json                   exists: True
v2_selection.json                  exists: True
v2_validation_folds.json           exists: True
v2_confirmation_apr.json           exists: True
v2_secondary_label.json            exists: True
v2_leakage_checks.json             exists: True
v2_freeze.json                     exists: True
v2_test.json                       exists: True
v2_playbook_summary.json           exists: True
v2_summary.json                    exists: True
v2_queue_2026-07.csv               exists: True
figures: ['fig1_fixed_position_ctr.png', 'fig2_forward_validation.png', 'fig3_validation_checks.png', 'fig4_ai_referrals.png', 'fig5_selection.png']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.